# JAXA catalog explorer

Browse the bundled JAXA catalog without touching the network. The catalog ships 118 STAC collections (the `jaxa-earth` protocol) plus six strategic G-Portal mission products (the `gportal` protocol), all driven by a `protocol` discriminator on each row.

> No credentials needed; no `[jaxa]` extra needed for this notebook either.

## Open the catalog

In [1]:
from earthlens.jaxa import Catalog

cat = Catalog()
len(cat)

917

## Split by protocol

The two protocols are routed independently by the backend. `by_protocol(...)` lists the canonical keys for each.

In [2]:
jaxa_earth_keys = cat.by_protocol('jaxa-earth')
gportal_keys = cat.by_protocol('gportal')
print(f'jaxa-earth: {len(jaxa_earth_keys)} collections')
print(f'gportal:    {len(gportal_keys)} products')
print()
print('First 5 jaxa-earth keys:')
for k in jaxa_earth_keys[:5]:
    print(f'  {k}')
print()
print('Every gportal key:')
for k in gportal_keys:
    print(f'  {k}')

jaxa-earth: 118 collections
gportal:    799 products

First 5 jaxa-earth keys:
  alos2-fnf
  amsr2-seaice-north-daily
  amsr2-seaice-south-daily
  amsr2-smc-d-daily
  amsr2-smc-d-halfmonth

Every gportal key:
  adeos-avnir-l1a-mu
  adeos-avnir-l1a-pan
  adeos-avnir-l1b2-mu
  adeos-avnir-l1b2-pan
  adeos-octs-gac-ocean-color
  adeos-octs-gac-ocean-color-3007
  adeos-octs-gac-ocean-color-chlorophyll-a
  adeos-octs-gac-ocean-color-czcs-like-pigm
  adeos-octs-gac-ocean-color1
  adeos-octs-gac-ocean-color2
  adeos-octs-gac-sea-surface-temperatur
  adeos-octs-gac-sea-surface-temperture
  adeos-octs-gac-seasurface-temperatur
  adeos-octs-gac-thermal-infrared
  adeos-octs-gac-vegetation-index
  adeos-octs-gac-vegetation-index-3001
  adeos-octs-gac-vegetation-index-3011
  adeos-octs-gac-visible-and-near-infrared
  adeos-octs-k490
  adeos-octs-k490-3010
  adeos-octs-rtc-ocean-color-chlorophyll-a
  adeos-octs-rtc-ocean-color-czcs-like-pigm
  adeos-octs-rtc-ocean-color1
  adeos-octs-rtc-ocean-colo

## Friendly aliases

Every one of the 118 jaxa-earth collections has a short, friendly canonical key — the long auto-derived slug stays as an alias for round-tripping. A handful of high-traffic products also accept plain-English aliases (`elevation`, `precipitation`, `lccs`, ...).

In [3]:
for alias in [
    'elevation',
    'dem',
    'precipitation',
    'lccs',
    'forest-non-forest',
    'palsar2',
    'earthcare',
    'gpm',
]:
    row = cat.get(alias)
    print(f'{alias:18s} -> {row.key:25s} ({row.protocol})')

elevation          -> aw3d30                    (jaxa-earth)
dem                -> aw3d30                    (jaxa-earth)
precipitation      -> gsmap                     (jaxa-earth)
lccs               -> proba-v-lccs              (jaxa-earth)
forest-non-forest  -> alos2-fnf                 (jaxa-earth)
palsar2            -> alos2-palsar2-uf-sp       (gportal)
earthcare          -> earthcare-cpr-eco         (gportal)
gpm                -> gpm-dpr-kupr-l1b          (gportal)


## Mission families at a glance

The friendly keys follow a `<mission>-<product>[-<d|n>][-<cadence>][-norm]` pattern. Group by the mission prefix to see how the 118 collections decompose.

In [4]:
from collections import Counter

families = Counter(k.split('-', 1)[0] for k in cat.by_protocol('jaxa-earth'))
for family, count in families.most_common():
    print(f'  {family:10s} {count:3d} collection(s)')

  sgli        39 collection(s)
  amsr2       20 collection(s)
  modis       19 collection(s)
  mod11        8 collection(s)
  gsmap        7 collection(s)
  temsm        7 collection(s)
  amsre        6 collection(s)
  myd11        4 collection(s)
  aw3d30       2 collection(s)
  mod11c3      2 collection(s)
  alos2        1 collection(s)
  hrlulc       1 collection(s)
  proba        1 collection(s)
  spi          1 collection(s)


## Inspect a row

Every row carries its protocol-specific identifier — a `collection` (STAC name) for `jaxa-earth` or a `short_name` (numeric id) for `gportal` — plus a default band where applicable.

In [5]:
aw3d30 = cat.get('elevation')
print('canonical key:', aw3d30.key)
print('protocol:     ', aw3d30.protocol)
print('collection:   ', aw3d30.collection)
print('default band: ', aw3d30.default_band)
print()
palsar2 = cat.get('palsar2')
print('canonical key:', palsar2.key)
print('protocol:     ', palsar2.protocol)
print('short_name:   ', palsar2.short_name)
print('description:  ', palsar2.description)

canonical key: aw3d30
protocol:      jaxa-earth
collection:    JAXA.EORC_ALOS.PRISM_AW3D30.v3.2_global
default band:  DSM

canonical key: alos2-palsar2-uf-sp
protocol:      gportal
short_name:    27004001
description:   ALOS-2 â€” NA/Ultra-fine[3m] SP


## Unknown keys raise with a did-you-mean hint

The catalog uses `difflib.get_close_matches` to suggest the nearest known key on a typo.

In [6]:
try:
    cat.get('aw3d3')
except ValueError as exc:
    print(exc)

'aw3d3' is not in the JAXA catalog. Known keys: ['adeos-avnir-l1a-mu', 'adeos-avnir-l1a-pan', 'adeos-avnir-l1b2-mu', 'adeos-avnir-l1b2-pan', 'adeos-octs-gac-ocean-color', 'adeos-octs-gac-ocean-color-3007', 'adeos-octs-gac-ocean-color-chlorophyll-a', 'adeos-octs-gac-ocean-color-czcs-like-pigm', 'adeos-octs-gac-ocean-color1', 'adeos-octs-gac-ocean-color2', 'adeos-octs-gac-sea-surface-temperatur', 'adeos-octs-gac-sea-surface-temperture', 'adeos-octs-gac-seasurface-temperatur', 'adeos-octs-gac-thermal-infrared', 'adeos-octs-gac-vegetation-index', 'adeos-octs-gac-vegetation-index-3001', 'adeos-octs-gac-vegetation-index-3011', 'adeos-octs-gac-visible-and-near-infrared', 'adeos-octs-k490', 'adeos-octs-k490-3010', 'adeos-octs-rtc-ocean-color-chlorophyll-a', 'adeos-octs-rtc-ocean-color-czcs-like-pigm', 'adeos-octs-rtc-ocean-color1', 'adeos-octs-rtc-ocean-color2', 'adeos-octs-rtc-sea-surface-temperatur', 'adeos-octs-rtc-sea-surface-temperture', 'adeos-octs-rtc-thermal-infrared', 'adeos-octs-rtc-

## Refreshing against the live SDK universes

The CLI command `earthlens datasets refresh jaxa` walks `jaxa.earth.ImageCollectionList` (STAC) and `gportal.datasets()` (G-Portal) to diff the bundled YAML against the live IDs. Both SDKs are optional — the `[jaxa]` extra installs them.

```bash
earthlens datasets refresh jaxa
```